In [ ]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcraft.trainer import ThermodynamicTrainer
from imolcraft.trainer.dmff_utils import parser_dmffyaml
from imolcraft.trainer.loss import loss_thermodynamicperturbation
from functools import partial

/Users/srak/apl/iMolCRAFT/DMFF/dmff/admp/qeq.py:33: UserWarning: jaxopt not found, QEQ cannot be used.
  warnings.warn("jaxopt not found, QEQ cannot be used.")
Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.


In [2]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 1.2,
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 100000,
  'nstxout': 1000,
  'neff': 30,
  'anneal_totaltime': 0},
 'targets': {'density_gcm3': {'weight': 1.0, 'gt': 1.568},
  'La_A': {'weight': 1.0, 'gt': 36.82},
  'Lc_A': {'weight': 1.0, 'gt': 40.2152},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [3]:
# for test 
params["sampling"]["relax_steps"] = 1000
params["sampling"]["prod_steps"] = 2000

In [4]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [ ]:
trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="merged_supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charge", 'VirtualSite/weight'],
    label="2rdf_1adf_density",
)

In [6]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [ ]:
trainer.ffparams["NonbondedForce"]["charge"].shape

(22,)

In [ ]:
from imolcraft.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[12].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[13].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[14].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[15].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[16].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[17].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[18].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[19].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[20].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[21].set(0.0)
    vsite_weight = (grads["VirtualSite"]["weight"][1]+grads["VirtualSite"]["weight"][3])/2
    grads["VirtualSite"]["weight"] = grads["VirtualSite"]["weight"].at[0].set(-vsite_weight)
    grads["VirtualSite"]["weight"] = grads["VirtualSite"]["weight"].at[1].set(vsite_weight)
    grads["VirtualSite"]["weight"] = grads["VirtualSite"]["weight"].at[2].set(-vsite_weight)
    grads["VirtualSite"]["weight"] = grads["VirtualSite"]["weight"].at[3].set(vsite_weight)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9,10,11]], target_charges=[0.0])
    return ffparams

In [9]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [10]:
trainer.setup()

Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5131033477017082,0,--
== Start Production ==
2000,1.5085197296283204,160,0:00
3000,1.5097226579892458,168,0:00


/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element EP found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:253: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 2/2 [00:00<00:00, 13.69it/s]


In [11]:
neff_prev = trainer.neff
print("Initial neff:", neff_prev)
trainer.neff = [50]

Initial neff: [30]


In [12]:
trainer.fit(10, 2)

  0%|          | 0/2 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 2/2 [00:01<00:00,  1.84it/s]


Effective sample sizes:
  sample_0: 1
  Total: 1.7537803318636602
  0 -> Resample
Resampling sample_0... by xmlfiles/epoch-1.xml
Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5048598171608225,0,--
== Start Production ==
2000,1.5025467460385384,170,0:00
3000,1.4951490566185275,169,0:00


/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element EP found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:253: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 2/2 [00:00<00:00, 51.58it/s]


Epoch 0 completed in 23.73 seconds.
Loss:  1.2326163789056728
Best Loss:  1.2326163789056728 at epoch  0


  0%|          | 0/2 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 2/2 [00:00<00:00,  2.66it/s]


...

  0%|          | 0/2 [00:00<?, ?it/s]/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32 with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
100%|██████████| 2/2 [00:00<00:00,  2.91it/s]


Effective sample sizes:
  sample_0: 1
  Total: 1.6343337805104592
  0 -> Resample
Resampling sample_0... by xmlfiles/epoch-9.xml
Using anisonpt ensemble
Anisotropic pressure control
== Energy minimization ==
== Start Relaxation ==
#"Step","Density (g/mL)","Speed (ns/day)","Time Remaining"
1000,1.5056732860754756,0,--
== Start Production ==
2000,1.4945736422596778,167,0:00
3000,1.5010168357331959,168,0:00


/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element EP found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/Users/srak/miniconda3/envs/imc_dev/lib/python3.11/site-packages/MDAnalysis/coordinates/XDR.py:253: UserWarning: Reload offsets from trajectory
 ctime or size or n_atoms did not match
  warnings.warn(
100%|██████████| 2/2 [00:00<00:00, 49.58it/s]


Epoch 8 completed in 18.07 seconds.
Loss:  1.3133634610666534
Best Loss:  1.1972021647748765 at epoch  2


KeyboardInterrupt: 

In [ ]:
trainer.neff = neff_prev

In [ ]:
trainer.fit(490, 2)

In [ ]:
trainer.from_checkpoint(trainer_checkpoint="train_state.pkl",
                        ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
                        nums_ffxml=[2,1,1],
                        pdbfile="merged_supercell_bonds.pdb",
                        initial_ffxml="xmlfiles/epoch-4.xml",
                        loss_fn=lossfn,
                        opt_fftypes=["NonbondedForce/charge", 'VirtualSite/weight'],
                        )